# What's Actually in a Raw Spectrum (and Why You Can't Skip Preprocessing)
### 3.1 — Noise, Signal, and Why Preprocessing Exists

The number the instrument hands you is never just the chemistry. A raw spectrum is
the analyte signal you want **plus** a pile of things you don't: random noise, a
sloping baseline, slow drift, the occasional spike. Preprocessing is how you
separate the two — and it is the most consequential, least examined part of the
whole analysis. Get it right and a real band emerges; get it wrong and you either
bury real chemistry or invent some that was never there.

This opening lesson of Track 3 is about *why* the rest of the track exists. Before
we smooth (3.2), correct baselines (3.3), or pick peaks (3.4), it pays to look
honestly at what is in a raw measurement and which parts are signal versus nuisance.

> **One idea to hold onto:** a raw measurement is **signal + nuisance**.
> Preprocessing is the scientific judgment of separating them — and every choice
> can reveal real chemistry or fabricate it. It is not cosmetic cleanup.

**By the end of this notebook you will be able to:**

1. Name the nuisance components hiding in a raw spectrum: noise, baseline, drift, artifacts.
2. Tell *signal* (smooth, correlated, reproducible) from *nuisance* (random, broad, or one-off).
3. See how a nuisance changes a reported number — and why preprocessing is judgment, not decoration.

## 1. Build a raw spectrum from its parts

The honest way to understand a measurement is to assemble one from components we
*know*, then look at the sum the way the instrument would hand it to us. Our raw
spectrum is a true two-band signal, plus a sloping baseline, plus random noise,
plus a single cosmic-ray spike.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (6.5, 3.6), "axes.grid": True,
                     "grid.alpha": 0.3, "font.size": 10})

wl = np.linspace(400, 700, 601)        # wavelength axis (nm), 0.5 nm spacing

def gaussian(center, height, fwhm):
    sigma = fwhm / 2.355               # FWHM -> standard deviation
    return height * np.exp(-((wl - center) ** 2) / (2 * sigma ** 2))

# The true chemistry: two absorbance bands. This is what we WANT to measure.
true_signal = gaussian(520, 0.80, 40) + gaussian(610, 0.50, 60)

# Nuisance 1: a sloping instrument baseline (lamp, optics, cuvette).
baseline = 0.15 + 0.0009 * (wl - 400)

# Nuisance 2: random detector noise.
rng = np.random.default_rng(7)
noise = rng.normal(0.0, 0.02, size=wl.size)

# Nuisance 3: a single cosmic-ray spike (a one-off artifact).
raw = true_signal + baseline + noise
spike_i = np.argmin(np.abs(wl - 470))
raw[spike_i] += 0.6

print("a raw spectrum is the sum of:")
print("  true signal  : the chemistry we want")
print("  + baseline   : a slow instrumental background")
print("  + noise      : random detector jitter")
print("  + artifact   : the cosmic-ray spike at", int(wl[spike_i]), "nm")

## 2. See the layers

Plotting the true signal, the baseline, and the raw measurement together shows how
much of what you "measured" isn't chemistry at all.

In [ ]:
fig, ax = plt.subplots()
ax.plot(wl, raw, lw=0.8, color="#9aa0a6", label="raw measurement")
ax.plot(wl, baseline, lw=1.6, ls="--", color="#ea4335", label="baseline (nuisance)")
ax.plot(wl, true_signal + baseline.min(), lw=2.0, color="#1a73e8",
        label="true signal (offset for clarity)")
ax.annotate("cosmic-ray spike", xy=(wl[spike_i], raw[spike_i]),
            xytext=(490, raw[spike_i]), fontsize=9,
            arrowprops=dict(arrowstyle="->", color="#5f6368"))
ax.set_xlabel("wavelength (nm)")
ax.set_ylabel("absorbance (AU)")
ax.set_title("One raw spectrum, three kinds of nuisance")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

The two real bands are there, but they sit on a baseline that lifts and tilts the
whole spectrum, they're fuzzed by noise, and a spike masquerades as a sharp peak
near 470 nm. Reporting anything off the raw curve means reporting the nuisances too.

## 3. What separates signal from nuisance?

The components behave differently, and those differences are exactly what lets us
pull them apart later:

| | behaves like | how to recognize it |
|---|---|---|
| **Signal** | smooth, *correlated* point-to-point; **reproducible** across scans | a real band repeats in every replicate |
| **Noise** | random, *uncorrelated*; high-frequency | changes every scan; averages out |
| **Baseline / drift** | slow, broad, low-frequency | spans the whole spectrum; moves slowly |
| **Artifact (spike)** | sharp, large, *one-off* | doesn't reproduce; appears in one scan only |

The single most useful test is **reproducibility**: measure twice. Whatever repeats
is signal; whatever doesn't is noise or a one-off.

In [ ]:
# Two independent scans of the SAME sample: same signal + baseline, fresh noise.
scan_a = true_signal + baseline + rng.normal(0, 0.02, size=wl.size)
scan_b = true_signal + baseline + rng.normal(0, 0.02, size=wl.size)

# Their difference cancels everything reproducible (signal + baseline) and leaves
# only the noise -- a direct look at the nuisance that does NOT repeat.
difference = scan_a - scan_b
per_scan_noise = np.std(difference) / np.sqrt(2)   # diff of two scans -> /sqrt(2)

print("std of (scan A - scan B):", round(np.std(difference), 4), "AU")
print("implied per-scan noise  :", round(per_scan_noise, 4), "AU  (true value was 0.02)")

fig, ax = plt.subplots()
ax.plot(wl, scan_a, lw=0.8, color="#1a73e8", label="scan A")
ax.plot(wl, scan_b, lw=0.8, color="#e8710a", label="scan B")
ax.set_xlim(430, 470)                 # zoom: the bands line up, the jitter doesn't
ax.set_xlabel("wavelength (nm)")
ax.set_ylabel("absorbance (AU)")
ax.set_title("Two scans of one sample: signal repeats, noise doesn't")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

The two scans trace the same underlying shape but jitter differently — and their
difference recovers the noise level almost exactly. That reproducibility test is the
bedrock idea behind signal averaging, which gets a full treatment in 3.6.

## 4. Why it matters: a nuisance changes the number you report

This is the point of the whole track. Suppose you read the height of the 520 nm
band straight off the raw spectrum. The baseline is sitting under it, so your number
is too high — and nothing warns you.

In [ ]:
i_band = np.argmin(np.abs(wl - 520))
true_height = true_signal[i_band]

# Naive: read the raw height (includes the baseline sitting underneath).
raw_height = (true_signal + baseline)[i_band]

# Corrected: estimate a local baseline from two peak-free anchor points and subtract.
left = np.argmin(np.abs(wl - 430))
right = np.argmin(np.abs(wl - 690))
slope = ((true_signal + baseline)[right] - (true_signal + baseline)[left]) / (wl[right] - wl[left])
baseline_at_band = (true_signal + baseline)[left] + slope * (wl[i_band] - wl[left])
corrected_height = (true_signal + baseline)[i_band] - baseline_at_band

print(f"true band height        : {true_height:.3f} AU")
print(f"read off RAW spectrum   : {raw_height:.3f} AU  ({100*(raw_height/true_height-1):+.0f}%)")
print(f"after baseline subtract : {corrected_height:.3f} AU  ({100*(corrected_height/true_height-1):+.0f}%)")

Reading the raw spectrum overstates the band by tens of percent — a bias that would
flow straight into a calibration or a comparison. The baseline correction isn't
tidying; it's the difference between a right answer and a wrong one. *That* is why
preprocessing exists.

## 5. But preprocessing is a trade-off, not a free fix

Removing noise has a cost. The simplest smoother — a moving average — quiets the
noise but, pushed too far, flattens the very peaks you're measuring. Here is the
same band smoothed gently and aggressively.

In [ ]:
def moving_average(y, window):
    kernel = np.ones(window) / window       # uniform weights summing to 1
    return np.convolve(y, kernel, mode="same")

measured = true_signal + baseline + rng.normal(0, 0.02, size=wl.size)

fig, ax = plt.subplots()
ax.plot(wl, measured, lw=0.7, color="#dadce0", label="noisy")
ax.plot(wl, true_signal + baseline, lw=2.0, color="#1a73e8", label="truth")
for w, col in [(9, "#188038"), (61, "#a142f4")]:
    ax.plot(wl, moving_average(measured, w), lw=1.4, label=f"moving avg, w={w}")
ax.set_xlim(470, 570)
ax.set_xlabel("wavelength (nm)")
ax.set_ylabel("absorbance (AU)")
ax.set_title("Smoothing trades noise against resolution")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

i_band = np.argmin(np.abs(wl - 520))
for w in (9, 61):
    h = moving_average(measured, w)[i_band]
    print(f"window {w:2d}: band height {h:.3f} AU "
          f"({100*h/(true_signal+baseline)[i_band]:.0f}% of the noisy peak)")

The narrow window calms the noise while leaving the band intact; the wide one keeps
smoothing but now visibly shaves the peak down. There is no setting that removes all
noise and distorts nothing — only a judgement about how much of each you can accept.
The next lesson (3.2) makes that judgement quantitative with Savitzky–Golay.

## 6. Preprocessing is scientific judgment

Every step ahead in this track is a decision with consequences, not a cosmetic
default:

- **Subtract a baseline** and you change every height and area you report — correct
  it badly and you bias the result; correct it well and quantitation becomes honest.
- **Smooth** and you trade noise for resolution — too much erases real, narrow features.
- **Remove an artifact** and you must be sure it's an artifact, not a real sharp peak.

The same operations that *reveal* chemistry can *fabricate* it. So preprocessing
carries the same obligations as the rest of the measurement: choose deliberately,
justify the choice from what you can measure (noise level, peak width, baseline
shape), and **record every step** so the path from raw bytes to reported number can
be reproduced and defended.

## Key Takeaways

- A raw spectrum is **signal + noise + baseline/drift + artifacts** — most of what
  you "measured" may not be chemistry.
- **Signal is reproducible; noise is not.** Measuring twice is the fundamental test
  that separates them.
- A nuisance you ignore (like a baseline) **biases the number you report**, silently.
- Preprocessing is a **trade-off and a judgment**, not cosmetic cleanup — and it must
  be recorded.

## Practical Checklist

- [ ] Before processing, ask what in the raw signal is chemistry and what is nuisance.
- [ ] Use replicate scans to estimate the noise level and confirm features reproduce.
- [ ] Check whether a baseline is biasing the height or area you intend to report.
- [ ] Treat every preprocessing step as a decision with consequences, and write it down.

## Common Mistakes

- Reading heights or areas off a raw spectrum with the baseline still under them.
- Treating a one-off spike as a real peak (or smoothing it in) instead of recognizing an artifact.
- Over-smoothing until narrow, real features are gone.
- Regarding preprocessing as cosmetic, and not reporting what was done.

## Reporting Guidance

- State every preprocessing step and its parameters (baseline method, smoothing
  window, despiking) in your methods — they are part of how the number was produced.

## Next Lesson

**3.2 — Savitzky–Golay Smoothing and Derivatives.** You've seen the noise-vs-resolution
trade-off with a crude moving average; next we make smoothing a deliberate, quantitative
choice that preserves peak shape.